# KAT-VAD v2 — P6 / **O1′ on T2-test** (Amendment 8, Q4–Q5)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§17 Amendment 8**, written after batch 2 and before this number.
Batch 2's A2/A3 **FAILED O1 on T2-val** (micro > clip oracle, macro < micro) while window AUC **and** macro rose over A0.
O1′ calls an arm *collapsed* only when window AUC rises **and** macro falls below A0 − 0.01. Here the four pilot
checkpoints are scored on **T2-test** (1,106 windows, unseen by every v2 arm). **No training. No DoTA.**

| step | what | GPU? |
|---|---|---|
| 0 | setup | — |
| 1 | preflight: Amendment 8 in the upload, J10 on 4 ckpts, T2-val diags, stage the parent T2 dir + 4 input caches | no |
| 2 | `core.tools.v2_guard_test`: A0–A3 on T2-test (hard gate: every T2-test source in each cache), O1 + O1′, Q5 | optional (CPU ≈ minutes) |
| 3 | tool check (A0 here == Amendment 7's A0 on T2-test) → print → stop | — |

The registered O1 verdict on T2-val is **kept** (Q1); O1′ can only rescue an arm into E3. No adoption reads T2-test.
Everything is written under `Thesis-V2/outputs/v2_pilot/o1prime_t2test/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (plain CLIP)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/, cache/v2/ (baked), outputs/v2_pilot; write: outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 17. Amendment 8' in addendum, 'this upload predates Amendment 8 -- re-upload branch v2'
assert (REPO / 'core/tools/v2_guard_test.py').is_file(), 'this upload has no core/tools/v2_guard_test.py -- re-upload'

DATASET  = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
SEED     = 2099
CRN      = 'R2'
T2_TEST_WINDOWS = 1106
TOOL_TOL = 1e-3                                         # step 3: A0 here vs Amendment 7's A0 (same ckpt, same windows)

T2       = constants.DATA_ROOT / DATASET                # PARENT dir: frame_labels_test.json = T2-test windows
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET
V2_CACHE = Path(DRIVE_V2) / 'cache' / 'v2'
PILOT    = constants.OUTPUT_ROOT / 'v2_pilot'
OUT      = PILOT / 'o1prime_t2test'
ARMS     = ('A0', 'A1', 'A2', 'A3')
INPUT = {'A0': CLIP_DIR,                                # plain CLIP (A0 has no baked cache)
         'A1': V2_CACHE / f'A1_{CRN}' / DATASET,
         'A2': V2_CACHE / 'A2_S' / DATASET,
         'A3': V2_CACHE / f'A3_{CRN}_S' / DATASET}
CKPT = {a: PILOT / a / f's{SEED}' / 'checkpoint_last.pt' for a in ARMS}
VAL_DIAG = {a: PILOT / a / f's{SEED}' / 'diag_cap' / 'diag.json' for a in ARMS}   # batch 2 read (A0/A1 == batch 1)
D5_T2TEST = PILOT / 'd5_t2test' / 'd5_t2test.json'     # Amendment 7: A0's T2-test micro / macro via core.evaluate
LOCAL = Path('/content/o1p')
LOCAL.mkdir(parents=True, exist_ok=True)
for a in ARMS:
    print(a, CKPT[a], '|', INPUT[a])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU -- fine, eval only'

In [ ]:
import hashlib
import json
import shutil
import subprocess
import time

import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst):
    """Copy a Drive dir/file to VM disk once (bit-identical; C2 untouched)."""
    if dst.exists():
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive and fsync every file (C10)."""
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight + stage

In [ ]:
for path in (T2 / constants.FRAME_LABELS_TEST_FILENAME, D5_T2TEST, *INPUT.values(), *CKPT.values(), *VAL_DIAG.values()):
    assert path.exists(), f'missing {path}'

# J10 on every checkpoint: its own global_step == the last step of the metrics.jsonl beside it.
for a, ckpt in CKPT.items():
    step, last = ckpt_step(ckpt), metrics_last(ckpt.parent / 'metrics.jsonl')
    assert last is not None and step == last['global_step'], f'{a}: J10 FAILS (ckpt {step}, metrics {last})'
    print(f'{a}: step {step} (J10 ok)')

# The T2-val side is batch 2's read, as recorded in RESULTS_P6_PILOT.md.
RECORDED = {'A0': (0.6586, 0.6737), 'A1': (0.6683, 0.6705), 'A2': (0.7120, 0.6882), 'A3': (0.7161, 0.7052)}
for a in ARMS:
    g = json.loads(VAL_DIAG[a].read_text())['guardrails']
    assert (round(g['micro'], 4), round(g['macro'], 4)) == RECORDED[a], f'{a}: diag_cap != the recorded batch 2 read'
print('T2-val diags == recorded batch 2')

L_T2 = stage(T2, LOCAL / 'stage' / 'corpus')
test = json.loads((L_T2 / constants.FRAME_LABELS_TEST_FILENAME).read_text())
assert len(test) == T2_TEST_WINDOWS, f'{len(test)} T2-test windows, expected {T2_TEST_WINDOWS}'
windows = json.loads((L_T2 / 'windows.json').read_text())
test_sources = sorted({windows[w]['source'] for w in test})
L_IN = {}
for a, src in INPUT.items():                             # Q4 hard gate, checked here before copying anything
    missing = [s for s in test_sources if not (src / f'{s}.npy').exists()]
    assert not missing, (f'{a}: {len(missing)}/{len(test_sources)} T2-test sources missing in {src} '
                         f'(first {missing[:5]}) -- STOP, paste to Claude: re-apply, never refit')
    L_IN[a] = stage(src, LOCAL / 'stage' / f'{a}_in')
print(f'T2-test: {len(test)} windows from {len(test_sources)} sources; every cache covers them')

## 2. Score A0–A3 on T2-test — `core.tools.v2_guard_test`

Same forward and reads as `v2_diagnostics` (whole window, `score_norm=auto`, window AUC = max score vs window label).
Re-run the cell after a disconnect; a finished read-out on Drive is reused.

In [ ]:
from core.tools import v2_guard_test

if (OUT / v2_guard_test.GUARD_JSON).exists():
    print(f'reusing {OUT / v2_guard_test.GUARD_JSON}')
else:
    local = LOCAL / 'out'
    cmd = [sys.executable, '-m', 'core.tools.v2_guard_test', '--data-dir', L_T2, '--a0', 'A0',
           '--out-dir', local, '--device', 'auto']
    for a in ARMS:
        cmd += ['--arm', a, CKPT[a], L_IN[a], '--val-diag', a, VAL_DIAG[a]]
    run(cmd, log=LOCAL / 'guard_test.log')
    publish(local, OUT)
R = json.loads((OUT / v2_guard_test.GUARD_JSON).read_text())
for a in ARMS:
    assert R['runs'][a]['windows'] == T2_TEST_WINDOWS, f"{a}: scored {R['runs'][a]['windows']} windows"
print('scored', T2_TEST_WINDOWS, 'T2-test windows per arm')

## 3. Tool check → print → **stop**

In [ ]:
# Tool check (not a verdict): A0 through v2_guard_test must equal A0 through core.evaluate (Amendment 7, H2's row).
d5 = json.loads(D5_T2TEST.read_text())['t2_test']['A0_s2099']
a0 = R['t2_test']['A0']
delta = (a0['micro'] - d5[0], a0['macro'] - d5[1])
print(f'A0 T2-test here {a0["micro"]:.4f} / {a0["macro"]:.4f} vs Amendment 7 {d5[0]:.4f} / {d5[1]:.4f} '
      f'(Δ {delta[0]:+.5f} / {delta[1]:+.5f})')
assert all(abs(d) <= TOOL_TOL for d in delta), \
    'v2_guard_test does not reproduce core.evaluate on A0 -- STOP, paste this to Claude; do not read the verdict'

print((OUT / v2_guard_test.GUARD_MD).read_text())

Paste `v2_pilot/o1prime_t2test/o1prime_t2test.md` back to Claude. Whatever Q5 says, no recipe, `c`, `lr` or checkpoint
changes (Q1, Q5), and the T2-val O1 FAIL stays in every table beside this verdict.